In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input/chexpert'):
    print(root, len(files), "files")
    if len(files) > 0:
        break  # just peek at first non-empty folder to confirm structure

In [4]:
!git clone -b training https://github.com/zimcodes1/MEDISCAN.git /kaggle/working/ISNet_repo

fatal: destination path '/kaggle/working/ISNet_repo' already exists and is not an empty directory.


In [5]:
!cd /kaggle/working/ISNet_repo && git pull

Already up to date.


In [2]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/ISNet_repo")

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "https://github.com/PedroRASB/FasterISNet.git",
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print("Repository already exists — skipping clone.")

print("Repository:", REPO_DIR)
print("Code directory exists:", (REPO_DIR / "code").exists())

print("\nImportant files:")
for name in [
    "ISNetFlexTorch.py",
    "ISNetFlexLightning.py",
    "LRPDenseNetZe.py",
]:
    path = REPO_DIR / "code" / name
    print(f"{name}: {path.exists()}")

Cloning into '/kaggle/working/ISNet_repo'...


Repository: /kaggle/working/ISNet_repo
Code directory exists: True

Important files:
ISNetFlexTorch.py: True
ISNetFlexLightning.py: True
LRPDenseNetZe.py: True


In [3]:
import os

ISNET_ROOT = "/kaggle/working/ISNet_repo"

for root, dirs, files in os.walk(ISNET_ROOT):
    for file in files:
        if file.endswith(".py"):
            print(os.path.join(root, file))

/kaggle/working/ISNet_repo/code/LRPDenseNetZe.py
/kaggle/working/ISNet_repo/code/ISNetFlexLightning.py
/kaggle/working/ISNet_repo/code/globalsZe.py
/kaggle/working/ISNet_repo/code/RunISNet.py
/kaggle/working/ISNet_repo/code/SingleLabelEval.py
/kaggle/working/ISNet_repo/code/locations.py
/kaggle/working/ISNet_repo/code/ISNetLightningZe.py
/kaggle/working/ISNet_repo/code/ISNetFunctionsZe.py
/kaggle/working/ISNet_repo/code/RunISNetGrad.py
/kaggle/working/ISNet_repo/code/ISNetLayersZe.py
/kaggle/working/ISNet_repo/code/ISNetFlexTorch.py
/kaggle/working/ISNet_repo/code/compare_auc_delong_xu.py
/kaggle/working/ISNet_repo/code/RunISNetFlex.py
/kaggle/working/ISNet_repo/code/ISNetLightningZeGradient.py
/kaggle/working/ISNet_repo/code/resnet.py


In [6]:
from pathlib import Path
import shutil
import importlib
import inspect
import ISNetFlexTorch

# --------------------------------------------------
# 1. Locate the Kaggle module
# --------------------------------------------------
module_path = Path(ISNetFlexTorch.__file__)
print("Patching:", module_path)

# --------------------------------------------------
# 2. Safety backup
# --------------------------------------------------
backup_path = module_path.with_suffix(".py.before_bce_patch")

if not backup_path.exists():
    shutil.copy2(module_path, backup_path)
    print("Backup created:", backup_path)
else:
    print("Backup already exists:", backup_path)

# --------------------------------------------------
# 3. Read current source
# --------------------------------------------------
source = module_path.read_text()

old_block = """    if len(labels.shape)>1:
        labels=labels.squeeze(-1)
    if len(labels.shape)>1:
        L['classification']=F.binary_cross_entropy_with_logits(outputs,labels)
    else:
        L['classification']=torch.nn.functional.cross_entropy(outputs,labels)
"""

new_block = """    if outputs.shape[-1] == 1:
        labels = labels.float()

        if labels.dim() == 1:
            labels = labels.unsqueeze(-1)

        L['classification'] = F.binary_cross_entropy_with_logits(
            outputs,
            labels
        )
    else:
        if labels.dim() > 1:
            labels = labels.squeeze(-1)

        labels = labels.long()

        L['classification'] = F.cross_entropy(
            outputs,
            labels
        )
"""

# --------------------------------------------------
# 4. Verify expected original code exists
# --------------------------------------------------
if old_block not in source:
    raise RuntimeError(
        "The expected original classification block was not found. "
        "No changes were made."
    )

# --------------------------------------------------
# 5. Apply patch exactly once
# --------------------------------------------------
patched_source = source.replace(old_block, new_block, 1)

module_path.write_text(patched_source)

print("Classification-loss patch applied successfully.")

# --------------------------------------------------
# 6. Reload module
# --------------------------------------------------
importlib.invalidate_caches()
importlib.reload(ISNetFlexTorch)

print("ISNetFlexTorch reloaded.")

# --------------------------------------------------
# 7. Verify
# --------------------------------------------------
src = inspect.getsource(ISNetFlexTorch.CompoundLoss)

start = src.find("outputs=out['output']")
end = src.find("if LRPFlex is not None:")

print("\n===== PATCHED CLASSIFICATION SECTION =====")
print(src[start:end])
print("==========================================")

ModuleNotFoundError: No module named 'ISNetFlexTorch'

In [7]:
from pathlib import Path
import sys

ISNET_CODE = Path("/kaggle/working/ISNet_repo/code")

print("ISNet code directory exists:", ISNET_CODE.exists())
print("ISNet code directory:", ISNET_CODE)

if ISNET_CODE.exists():
    print("\nRelevant files:")
    for f in ISNET_CODE.glob("*.py"):
        print(" ", f.name)

    if str(ISNET_CODE) not in sys.path:
        sys.path.insert(0, str(ISNET_CODE))

print("\nPython path updated.")
print("ISNet code in sys.path:", str(ISNET_CODE) in sys.path)

ISNet code directory exists: True
ISNet code directory: /kaggle/working/ISNet_repo/code

Relevant files:
  LRPDenseNetZe.py
  ISNetFlexLightning.py
  globalsZe.py
  RunISNet.py
  SingleLabelEval.py
  locations.py
  ISNetLightningZe.py
  ISNetFunctionsZe.py
  RunISNetGrad.py
  ISNetLayersZe.py
  ISNetFlexTorch.py
  compare_auc_delong_xu.py
  RunISNetFlex.py
  ISNetLightningZeGradient.py
  resnet.py

Python path updated.
ISNet code in sys.path: True


In [8]:
import ISNetFlexTorch

print("Imported successfully.")
print("Loaded from:")
print(ISNetFlexTorch.__file__)

Imported successfully.
Loaded from:
/kaggle/working/ISNet_repo/code/ISNetFlexTorch.py


In [9]:
from pathlib import Path
import shutil
import importlib
import inspect
import ISNetFlexTorch

# --------------------------------------------------
# 1. Locate the module
# --------------------------------------------------
module_path = Path(ISNetFlexTorch.__file__)
print("Patching:", module_path)

# --------------------------------------------------
# 2. Create safety backup
# --------------------------------------------------
backup_path = module_path.with_suffix(".py.before_bce_patch")

if not backup_path.exists():
    shutil.copy2(module_path, backup_path)
    print("Backup created:", backup_path)
else:
    print("Backup already exists:", backup_path)

# --------------------------------------------------
# 3. Read current source
# --------------------------------------------------
source = module_path.read_text()

old_block = """    if len(labels.shape)>1:
        labels=labels.squeeze(-1)
    if len(labels.shape)>1:
        L['classification']=F.binary_cross_entropy_with_logits(outputs,labels)
    else:
        L['classification']=torch.nn.functional.cross_entropy(outputs,labels)
"""

new_block = """    if outputs.shape[-1] == 1:
        labels = labels.float()

        if labels.dim() == 1:
            labels = labels.unsqueeze(-1)

        L['classification'] = F.binary_cross_entropy_with_logits(
            outputs,
            labels
        )
    else:
        if labels.dim() > 1:
            labels = labels.squeeze(-1)

        labels = labels.long()

        L['classification'] = F.cross_entropy(
            outputs,
            labels
        )
"""

# --------------------------------------------------
# 4. Verify original code exists
# --------------------------------------------------
if old_block not in source:
    raise RuntimeError(
        "The expected original classification block was not found. "
        "No changes were made."
    )

# --------------------------------------------------
# 5. Apply patch exactly once
# --------------------------------------------------
patched_source = source.replace(old_block, new_block, 1)
module_path.write_text(patched_source)

print("Classification-loss patch applied successfully.")

# --------------------------------------------------
# 6. Reload module
# --------------------------------------------------
importlib.invalidate_caches()
importlib.reload(ISNetFlexTorch)

print("ISNetFlexTorch reloaded.")

# --------------------------------------------------
# 7. Verify patched source
# --------------------------------------------------
src = inspect.getsource(ISNetFlexTorch.CompoundLoss)

start = src.find("outputs=out['output']")
end = src.find("if LRPFlex is not None:")

print("\n===== PATCHED CLASSIFICATION SECTION =====")
print(src[start:end])
print("==========================================")

Patching: /kaggle/working/ISNet_repo/code/ISNetFlexTorch.py
Backup created: /kaggle/working/ISNet_repo/code/ISNetFlexTorch.py.before_bce_patch
Classification-loss patch applied successfully.
ISNetFlexTorch reloaded.

===== PATCHED CLASSIFICATION SECTION =====
outputs=out['output']
    LRPFlex=out['LRPFlex']
    
    if outputs.shape[-1] == 1:
        labels = labels.float()

        if labels.dim() == 1:
            labels = labels.unsqueeze(-1)

        L['classification'] = F.binary_cross_entropy_with_logits(
            outputs,
            labels
        )
    else:
        if labels.dim() > 1:
            labels = labels.squeeze(-1)

        labels = labels.long()

        L['classification'] = F.cross_entropy(
            outputs,
            labels
        )
    
    


In [10]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Kaggle inputs:")
for item in INPUT_ROOT.iterdir():
    print(" ", item)

Kaggle inputs:
  /kaggle/input/datasets


In [13]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets")

print("Top-level directories:")
for p in DATA_ROOT.iterdir():
    print(" ", p)

Top-level directories:
  /kaggle/input/datasets/ashery
  /kaggle/input/datasets/rhishamah


In [14]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/rhishamah")

print("Dataset contents:")
for item in DATA_ROOT.iterdir():
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind}: {item.name}")

Dataset contents:
DIR : mediscan-isnet-artifacts


In [15]:
from pathlib import Path

ARTIFACTS = Path("/kaggle/input/datasets/rhishamah/mediscan-isnet-artifacts")

print("Artifact contents:")
for item in ARTIFACTS.iterdir():
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind}: {item.name}")

Artifact contents:
DIR : lung_masks-20260924T090042Z-1-001
FILE: last.ckpt
FILE: split.json


In [16]:
import json
from pathlib import Path

SPLIT_PATH = Path(
    "/kaggle/input/datasets/rhishamah/"
    "mediscan-isnet-artifacts/split.json"
)

with open(SPLIT_PATH, "r") as f:
    split_data = json.load(f)

print("Type:", type(split_data))

if isinstance(split_data, dict):
    print("Keys:", list(split_data.keys()))

    for key, value in split_data.items():
        print(f"\n--- {key} ---")
        print("Type:", type(value))
        print("Length:", len(value) if hasattr(value, "__len__") else "N/A")
        print("First item:", value[0] if isinstance(value, list) and value else value)

Type: <class 'dict'>
Keys: ['train', 'val', 'test']

--- train ---
Type: <class 'list'>
Length: 14740
First item: ['/content/chexpert_data/train/patient00085/study1/view1_frontal.jpg', 1]

--- val ---
Type: <class 'list'>
Length: 1840
First item: ['/content/chexpert_data/train/patient00545/study1/view1_frontal.jpg', 1]

--- test ---
Type: <class 'list'>
Length: 1816
First item: ['/content/chexpert_data/train/patient00114/study15/view1_frontal.jpg', 1]


In [17]:
from pathlib import Path

CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery")

print("Top-level contents of ashery:")
for item in CHEXPERT_ROOT.iterdir():
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind}: {item.name}")

Top-level contents of ashery:
DIR : chexpert


In [18]:
from pathlib import Path

CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")

print("CheXpert contents:")
for item in CHEXPERT_ROOT.iterdir():
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind}: {item.name}")

CheXpert contents:
FILE: valid.csv
DIR : valid
FILE: train.csv
DIR : train


In [19]:
from pathlib import Path
import cv2

old_path = "/content/chexpert_data/train/patient00085/study1/view1_frontal.jpg"

relative_path = old_path.replace("/content/chexpert_data/", "")
kaggle_path = Path("/kaggle/input/datasets/ashery/chexpert") / relative_path

print("Mapped path:")
print(kaggle_path)

print("\nExists:", kaggle_path.exists())

if kaggle_path.exists():
    img = cv2.imread(str(kaggle_path), cv2.IMREAD_GRAYSCALE)
    print("Image loaded:", img is not None)
    if img is not None:
        print("Shape:", img.shape)

Mapped path:
/kaggle/input/datasets/ashery/chexpert/train/patient00085/study1/view1_frontal.jpg

Exists: True
Image loaded: True
Shape: (320, 390)


In [20]:
import json
from pathlib import Path

MASK_ROOT = Path(
    "/kaggle/input/datasets/rhishamah/"
    "mediscan-isnet-artifacts/lung_masks-20260924T090042Z-1-001"
)

# Look only for the mapping file in the mask folder's immediate contents
print("Mask folder contents:")
for item in MASK_ROOT.iterdir():
    print(" ", item.name)

Mask folder contents:
  lung_masks


In [21]:
from pathlib import Path

MASK_DIR = Path(
    "/kaggle/input/datasets/rhishamah/"
    "mediscan-isnet-artifacts/lung_masks-20260924T090042Z-1-001/"
    "lung_masks"
)

print("lung_masks contents:")
for item in MASK_DIR.iterdir():
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind}: {item.name}")

lung_masks contents:
FILE: 003769.npy
FILE: 007336.npy
FILE: 001314.npy
FILE: 010826.npy
FILE: 006394.npy
FILE: 002909.npy
FILE: 004756.npy
FILE: 000983.npy
FILE: 013468.npy
FILE: 002724.npy
FILE: 004602.npy
FILE: 003657.npy
FILE: 001467.npy
FILE: 007775.npy
FILE: 011624.npy
FILE: 012355.npy
FILE: 012754.npy
FILE: 001659.npy
FILE: 000708.npy
FILE: 011728.npy
FILE: 004684.npy
FILE: 013435.npy
FILE: 013704.npy
FILE: 014509.npy
FILE: 014211.npy
FILE: 008643.npy
FILE: 012360.npy
FILE: 001918.npy
FILE: 012782.npy
FILE: 011308.npy
FILE: 009550.npy
FILE: 008487.npy
FILE: 007123.npy
FILE: 010333.npy
FILE: 013765.npy
FILE: 011160.npy
FILE: 014158.npy
FILE: 008422.npy
FILE: 008366.npy
FILE: 002097.npy
FILE: 011621.npy
FILE: 004242.npy
FILE: 007632.npy
FILE: 008875.npy
FILE: 006384.npy
FILE: 007100.npy
FILE: 001194.npy
FILE: 010943.npy
FILE: 007810.npy
FILE: 008145.npy
FILE: 008091.npy
FILE: 013612.npy
FILE: 006247.npy
FILE: 013050.npy
FILE: 013170.npy
FILE: 011152.npy
FILE: 009043.npy
FILE: 0078

downloading pretrained model weights 